# Ministral-3-3B-Instruct-2512 — OCR trên Colab T4

Chạy toàn trang trên **cùng 43 ảnh, ID và hash** với Qwen/PaddleOCR/VietOCR.
Notebook dùng bản **BF16 chính thức của Mistral**, nạp thành **NF4 4-bit, compute FP16** cho T4,
không chạy kernel FP8. Mỗi ảnh được checkpoint và có log mỗi 30 giây.

ZIP chứa `outputs/ministral3_3b_colab_t4_nf4/{predictions.jsonl,run.json,texts/}`.
Chấm CER/WER bằng ground truth đã duyệt ở máy local; không tải nhãn từ GitHub.
Bật Google Drive nếu cần giữ kết quả qua các phiên; thay cấu hình thì dùng tên lượt chạy mới.

[Checkpoint gốc](https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512),
[BF16 chính thức](https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512-BF16),
[Transformers Mistral3](https://huggingface.co/docs/transformers/v5.3.0/model_doc/mistral3).


## 1. Môi trường riêng trên Colab T4

Chọn **T4 GPU** và chạy từ trên xuống trong một session riêng, sau khi đã lưu kết quả mô hình trước.
Không cài đè notebook này vào session Qwen đang chạy: phiên bản Transformers khác nhau.
Giữ PyTorch/CUDA của Colab, không cài requirements OCR local.
Nếu đã import Transformers trước ô cài thư viện, restart session sau khi cài rồi chạy tiếp từ ô cấu hình.


In [ ]:
import subprocess
import sys
PINS = ['transformers==5.3.0', 'accelerate==1.12.0', 'huggingface-hub==1.5.0', 'Pillow==11.3.0', 'requests>=2.32,<3', 'bitsandbytes==0.48.2', 'mistral-common==1.9.1']
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PINS])
print("Đã cài thư viện. Tiếp tục với ô cấu hình.")


In [ ]:
from pathlib import Path
REPO_URL = "https://github.com/datpq-alpha/ocr"
DATA_COMMIT = "eab241b9cd2c43c81255c247d81c49e83bcca04a"
EXPECTED_MANIFEST_SHA256 = "f64b2d370874e49e949b043776ab6a0db4f04399365aedb572a072b37c29dc68"
WARMUP_NEW_TOKENS = 64
SEED = 42
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/ocr_baseline")
else:
    WORK_ROOT = Path("/content/ocr_baseline")
DATA_ROOT = Path("/content/ocr_baseline_data")

REQUESTED_MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512"
MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
MODEL_REVISION = "b6d637bef2393152b3da2b2fde72eecdee30557e"
ENGINE = "ministral3_3b"
LOAD_IN_4BIT = True
MAX_IMAGE_SIDE = 1024
MAX_NEW_TOKENS = 8192
RUN_NAME = "ministral3_3b_colab_t4_" + ("nf4" if LOAD_IN_4BIT else "fp16")
OUTPUT_DIR = WORK_ROOT / "outputs" / RUN_NAME
PROMPT = 'Transcribe all legible text in this document image verbatim. Preserve Vietnamese accents, capitalization, punctuation, dates and numeric separators. Read the page from top to bottom. For tables, read rows from top to bottom and cells from left to right; finish all wrapped lines inside a cell before the next cell. Separate table cells with a tab and rows with a newline. Include headers, footers, stamps and handwritten text when legible. Do not correct spelling, translate, summarize, infer missing values or add explanations. Do not output Markdown fences or table formatting. Return only the transcribed text.'

assert MAX_IMAGE_SIDE > 0 and MAX_NEW_TOKENS > 0
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in (".", "..")
print("Output:", OUTPUT_DIR)


In [ ]:
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import time
import traceback
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import PurePosixPath


def canonical_sha256(value):
    encoded = json.dumps(value, ensure_ascii=False, sort_keys=True,
                         separators=(",", ":"), allow_nan=False).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def atomic_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp-" + uuid.uuid4().hex)
    try:
        with temporary.open("w", encoding="utf-8", newline="\n") as stream:
            stream.write(text)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def atomic_json(path, value):
    atomic_text(path, json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n")


def percentile(values, proportion):
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * proportion
    lower, upper = math.floor(position), math.ceil(position)
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)


def validate_samples(manifest):
    ids, paths = set(), set()
    samples = manifest["samples"]
    if not samples:
        raise ValueError("Empty manifest")
    for sample in samples:
        key, relative = sample["sample_id"], sample["image_path"]
        path = PurePosixPath(relative)
        if not re.fullmatch(r"GT\d{6}", key) or key in ids:
            raise ValueError(f"Invalid/duplicate ID: {key}")
        if (path.is_absolute() or ".." in path.parts or "\\" in relative
                or len(path.parts) < 3 or path.parts[0] != "Accounting_Documents"
                or path.parts[1] not in ("02_Accounting_Vouchers", "03_Invoices")
                or relative in paths):
            raise ValueError(f"Invalid/duplicate image path: {relative}")
        if not re.fullmatch(r"[0-9a-f]{64}", sample["sha256"]):
            raise ValueError(f"Invalid image hash: {key}")
        ids.add(key)
        paths.add(relative)
    return samples


def validate_row(sample, row, fingerprint):
    if (row.get("sample_id") != sample["sample_id"]
            or row.get("image_path") != sample["image_path"]
            or row.get("image_sha256") != sample["sha256"]
            or row.get("run_fingerprint") != fingerprint):
        raise ValueError(f"Saved prediction identity mismatch: {sample['sample_id']}")
    if row.get("status") not in ("Success", "Failed"):
        raise ValueError("Invalid prediction status")
    if row["status"] == "Success":
        if not isinstance(row.get("text"), str) or row["timing_seconds"]["total"] < 0:
            raise ValueError("Invalid text/timing")


def load_records(output_dir, samples, fingerprint):
    records = {}
    expected = {s["sample_id"]: s for s in samples}
    for path in (output_dir / "records").glob("*.json"):
        if path.stem not in expected:
            raise ValueError(f"Unexpected saved sample: {path.name}")
        row = json.loads(path.read_text(encoding="utf-8"))
        validate_row(expected[path.stem], row, fingerprint)
        records[path.stem] = row
    return records


def write_artifacts(output_dir, samples, records, metadata):
    rows = [records[s["sample_id"]] for s in samples if s["sample_id"] in records]
    atomic_text(output_dir / "predictions.jsonl", "".join(
        json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n" for row in rows))
    for row in rows:
        if row["status"] == "Success":
            text_path = output_dir / "texts" / (row["sample_id"] + ".txt")
            if not text_path.exists():
                atomic_text(text_path, row["text"] + "\n")
            elif text_path.read_text(encoding="utf-8") != row["text"] + "\n":
                raise ValueError(f"Saved text was edited: {text_path}; use a new RUN_NAME")
    successful = [row for row in rows if row["status"] == "Success"]
    times = [row["timing_seconds"]["total"] for row in successful]
    failed = len(rows) - len(successful)
    metadata.update(
        completed_images=len(successful), failed_images=failed,
        pending_images=len(samples) - len(rows),
        failure_rate=failed / len(rows) if rows else 0.0,
        truncated_images=sum(row.get("truncated", False) for row in successful),
        latency_p50_seconds=percentile(times, 0.5),
        latency_p95_seconds=percentile(times, 0.95),
        mean_image_seconds=sum(times) / len(times) if times else None,
        wall_seconds=sum(s["wall_seconds"] for s in metadata.get("sessions", [])),
        initialization_seconds=sum(s["initialization_seconds"] for s in metadata.get("sessions", [])),
        warmup_seconds=sum(s["warmup_seconds"] for s in metadata.get("sessions", [])),
        peak_gpu_allocated_bytes=max((r.get("gpu_memory", {}).get("peak_allocated_bytes", 0)
                                      for r in rows), default=0),
        peak_gpu_reserved_bytes=max((r.get("gpu_memory", {}).get("peak_reserved_bytes", 0)
                                     for r in rows), default=0),
    )
    wall = metadata["wall_seconds"]
    metadata["throughput_images_per_minute"] = len(successful) * 60 / wall if wall else None
    atomic_json(output_dir / "run.json", metadata)


def run_benchmark(samples, output_dir, identity, session_info, predict_fn):
    """Checkpoint each image; resume missing IDs without mixing configurations."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "records").mkdir(exist_ok=True)
    (output_dir / "texts").mkdir(exist_ok=True)
    fingerprint = canonical_sha256(identity)
    run_path = output_dir / "run.json"
    if run_path.exists():
        metadata = json.loads(run_path.read_text(encoding="utf-8"))
        if metadata.get("run_fingerprint") != fingerprint:
            raise ValueError("Configuration/data/runtime changed; choose a new RUN_NAME")
    else:
        if any((output_dir / "records").iterdir()) or (output_dir / "predictions.jsonl").exists():
            raise ValueError("Existing output without provenance; choose a new RUN_NAME")
        metadata = dict(identity)
        metadata.update(created_at=datetime.now(timezone.utc).isoformat(),
                        run_fingerprint=fingerprint, sessions=[])
    records = load_records(output_dir, samples, fingerprint)
    # Rebuild the exported JSONL from atomic per-image records after interruptions.
    write_artifacts(output_dir, samples, records, metadata)
    pending = [s for s in samples if s["sample_id"] not in records]
    if not pending:
        print("No missing images. Existing results retained.")
        return metadata
    session = dict(session_info, session_id=uuid.uuid4().hex,
                   started_at=datetime.now(timezone.utc).isoformat(), wall_seconds=0.0,
                   attempted_images=0, status="Running")
    metadata["sessions"].append(session)
    metadata["status"] = "Running"
    started = time.perf_counter()
    try:
        write_artifacts(output_dir, samples, records, metadata)
        for sample in pending:
            print(f"[START {len(records) + 1}/{len(samples)}] {sample['sample_id']}", flush=True)
            row = {"sample_id": sample["sample_id"], "image_path": sample["image_path"],
                   "image_sha256": sample["sha256"], "engine": identity["engine"],
                   "run_fingerprint": fingerprint, "session_id": session["session_id"]}
            attempt_started = time.perf_counter()
            try:
                row.update(predict_fn(sample), status="Success")
            except Exception:
                row.update(status="Failed", error=traceback.format_exc(),
                           timing_seconds={"total": time.perf_counter() - attempt_started})
                import gc
                gc.collect()
                if "torch" in globals() and torch.cuda.is_available():
                    torch.cuda.empty_cache()
            validate_row(sample, row, fingerprint)
            print(f"  {sample['sample_id']}: OCR {row['status']}; saving checkpoint"
                  f" ({row['timing_seconds']['total']:.1f}s"
                  + (f", {row['generated_tokens']} tokens" if "generated_tokens" in row else "")
                  + ")", flush=True)
            atomic_json(output_dir / "records" / (sample["sample_id"] + ".json"), row)
            records[sample["sample_id"]] = row
            session["attempted_images"] += 1
            session["wall_seconds"] = time.perf_counter() - started
            write_artifacts(output_dir, samples, records, metadata)
            print(f"[{len(records)}/{len(samples)}] {sample['sample_id']}: {row['status']}"
                  + (" (TOKEN LIMIT)" if row.get("truncated") else ""), flush=True)
    finally:
        session["wall_seconds"] = time.perf_counter() - started
        session["status"] = "Completed" if len(records) == len(samples) else "Interrupted"
        metadata["status"] = ("Interrupted" if len(records) < len(samples) else
                              "CompletedWithErrors" if any(r["status"] == "Failed" for r in records.values())
                              else "Completed")
        write_artifacts(output_dir, samples, records, metadata)
    return metadata


## 2. Ảnh từ GitHub

Cố định commit và fingerprint của manifest đang khớp với máy local. Mỗi ảnh được kiểm tra
SHA-256 trước khi nhận dạng. Chỉ tải manifest và ảnh, không tải texts/annotations của ground truth.
Khác biệt CRLF/LF của manifest không ảnh hưởng fingerprint nội dung JSON.


In [ ]:
from urllib.parse import quote
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

http = requests.Session()
http.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"])))
raw_base = f"https://raw.githubusercontent.com/datpq-alpha/ocr/{DATA_COMMIT}/"
response = http.get(raw_base + "ground_truth/manifest.json", timeout=(15, 90))
response.raise_for_status()
manifest = response.json()
if canonical_sha256(manifest) != EXPECTED_MANIFEST_SHA256:
    raise ValueError("Manifest differs from the local baseline; check DATA_COMMIT/fingerprint")
samples = validate_samples(manifest)
if len(samples) != 43:
    raise ValueError("Expected the same 43 images as the local baseline")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
atomic_json(DATA_ROOT / "manifest.json", manifest)
for sample in samples:
    destination = DATA_ROOT / sample["image_path"]
    if not destination.exists() or file_sha256(destination) != sample["sha256"]:
        response = http.get(raw_base + quote(sample["image_path"], safe="/"), timeout=(15, 90))
        response.raise_for_status()
        if hashlib.sha256(response.content).hexdigest() != sample["sha256"]:
            raise ValueError(f"Downloaded image hash mismatch: {sample['sample_id']}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(response.content)
    if file_sha256(destination) != sample["sha256"]:
        raise ValueError(f"Image hash mismatch: {sample['sample_id']}")
print("Verified images:", len(samples), dict(Counter(s["category"] for s in samples)))


## 3. Ministral: trọng số BF16 chính thức → NF4, tính toán FP16

Checkpoint gốc `Ministral-3-3B-Instruct-2512` là FP8. Notebook nạp bản
`Ministral-3-3B-Instruct-2512-BF16` do **Mistral phát hành**, sau đó lượng tử hóa NF4
các lớp ngôn ngữ và tính toán FP16. Vision/projector không lượng tử hóa.
Không yêu cầu T4 chạy BF16 hoặc kernel FP8. Đây là cấu hình khác bản FP8, cần ghi rõ trong báo cáo.

Giữ tỉ lệ ảnh; giới hạn cạnh dài 1.024 pixel được truyền cho PixtralProcessor.
Prompt chép toàn trang giống notebook Qwen; không dùng ground truth trong suy luận.
Không có API thương mại hoặc khóa API.


In [ ]:
import gc
import random
import torch
from PIL import Image, ImageOps
import transformers
if transformers.__version__ != "5.3.0":
    raise RuntimeError("Restart session sau ô cài thư viện; cần Transformers 5.3.0.")
if not torch.cuda.is_available():
    raise RuntimeError("Chọn T4 GPU trong Runtime → Change runtime type.")
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.backends.cuda.matmul.allow_tf32 = False
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; VRAM: {gpu.total_memory / 2**30:.2f} GiB; CUDA: {torch.version.cuda}")
if "T4" not in gpu.name:
    print("GPU khác T4; phần cứng thực tế được lưu để báo cáo riêng.")

from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
load_started = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
                                          trust_remote_code=False)
load_options = dict(revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float16,
                    device_map={"": 0}, attn_implementation="sdpa", low_cpu_mem_usage=True)
skip_modules = ["vision_tower", "multi_modal_projector", "lm_head"]
if LOAD_IN_4BIT:
    load_options["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16, llm_int8_skip_modules=skip_modules)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_options).eval()
torch.cuda.synchronize()
initialization_seconds = time.perf_counter() - load_started
quantized_names = []
if LOAD_IN_4BIT:
    import bitsandbytes as bnb
    quantized_names = [name for name, module in model.named_modules()
                       if isinstance(module, bnb.nn.Linear4bit)]
    if not quantized_names or any("vision_tower" in name or "multi_modal_projector" in name
                                  for name in quantized_names):
        raise RuntimeError("Unexpected quantization layout; inspect before benchmarking")
packages = {name: importlib.metadata.version(name) for name in
            ("torch", "torchvision", "transformers", "accelerate", "bitsandbytes",
             "huggingface-hub", "Pillow", "requests", "mistral-common")}
configuration = {
    "model_id": REQUESTED_MODEL_ID, "requested_model_id": REQUESTED_MODEL_ID, "weights_id": MODEL_ID,
    "model_revision": MODEL_REVISION, "source_precision": "BF16 official checkpoint",
    "quantization": "bitsandbytes_nf4_double_quant" if LOAD_IN_4BIT else "none",
    "compute_dtype": "float16", "vision_skip_modules": skip_modules,
    "quantized_module_names": quantized_names,
    "attention_backend": "sdpa", "max_image_side": MAX_IMAGE_SIDE,
    "max_new_tokens": MAX_NEW_TOKENS, "prompt": PROMPT,
    "preprocessing": "EXIF transpose; alpha on white; RGB; Pixtral aspect-preserving longest-edge resize",
    "reading_order": "full-page transcription; row-major tables; wrapped lines within each cell",
}

configuration["prompt_sha256"] = hashlib.sha256(PROMPT.encode("utf-8")).hexdigest()
configuration.update(batch_size=1, seed=SEED, do_sample=False, num_beams=1,
                     warmup_images_per_session=1, warmup_max_new_tokens=WARMUP_NEW_TOKENS,
                     measured_inference_passes_per_image=1,
                     cpu_threads=torch.get_num_threads())
identity = {
    "engine": ENGINE, "device": "cuda:0", "python": sys.version,
    "platform": platform.platform(), "packages": packages, "configuration": configuration,
    "hardware": {"gpu": gpu.name, "gpu_total_memory_bytes": gpu.total_memory,
                 "compute_capability": list(torch.cuda.get_device_capability(0)),
                 "cuda": torch.version.cuda, "cpu_logical_count": os.cpu_count()},
    "source": {"repository_url": REPO_URL, "commit": DATA_COMMIT,
               "manifest_sha256": EXPECTED_MANIFEST_SHA256},
    "samples": [{k: s[k] for k in ("sample_id", "image_path", "sha256")} for s in samples],
    "timing_scope": "full image pipeline including decode/preprocess/GPU/generation/assembly; CUDA synchronized; excludes downloads/model load/warmup",
    "truncation_policy": "retain partial text as Success with truncated=true; include in CER/WER",
    "measured_inference_passes_per_image": 1,
    "initialization_includes_uncached_model_download": True,
}
print("Initialization:", round(initialization_seconds, 2), "seconds")


In [ ]:
import threading


class ImageProgress:
    """Only observe progress; never change generated tokens or stopping limits."""
    def __init__(self, sample_id, interval=30):
        self.sample_id = sample_id
        self.interval = interval
        self.stage = "starting"
        self.generated_tokens = 0
        self.last_token_at = None
        self.stop_event = threading.Event()

    def update(self, stage, generated_tokens=None):
        self.stage = stage
        if generated_tokens is not None and generated_tokens > self.generated_tokens:
            self.generated_tokens = generated_tokens
            self.last_token_at = time.perf_counter()

    def token_observer(self, prompt_tokens):
        def observe(input_ids, scores, **kwargs):
            self.update("generation", int(input_ids.shape[-1]) - prompt_tokens)
            return False  # Observe only: EOS and max_new_tokens still decide when to stop.
        return observe

    def emit(self):
        elapsed = time.perf_counter() - self.started
        suffix = (f"; last new token {time.perf_counter() - self.last_token_at:.0f}s ago"
                  if self.last_token_at is not None else "; waiting for first generated token")
        print(f"  [{self.sample_id}] {elapsed:.0f}s; stage={self.stage}; "
              f"generated={self.generated_tokens} tokens" + suffix, flush=True)

    def _watch(self):
        while not self.stop_event.wait(self.interval):
            self.emit()

    def __enter__(self):
        self.started = time.perf_counter()
        self.thread = threading.Thread(target=self._watch, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc_value, exc_tb):
        self.stop_event.set()
        self.thread.join(timeout=1)


def predict_image(sample, max_new_tokens=MAX_NEW_TOKENS):
    with ImageProgress(sample["sample_id"]) as progress:
        return _predict_image(sample, max_new_tokens, progress)


def _predict_image(sample, max_new_tokens, progress):
    progress.update("verify image / GPU sync")
    image_path = DATA_ROOT / sample["image_path"]
    if file_sha256(image_path) != sample["sha256"]:
        raise ValueError("Image changed after download")
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    progress.update("decode image")
    with Image.open(image_path) as opened:
        rgba = ImageOps.exif_transpose(opened).convert("RGBA")
        white = Image.new("RGBA", rgba.size, "white")
        white.alpha_composite(rgba)
        image = white.convert("RGB")
    decoded = time.perf_counter()
    progress.update("preprocess image")
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": PROMPT}]}]
    chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[chat], images=[image], return_tensors="pt",
                       images_kwargs={"size": {"longest_edge": MAX_IMAGE_SIDE}})
    inputs.pop("token_type_ids", None)
    if "image_sizes" not in inputs or "pixel_values" not in inputs:
        raise RuntimeError("Processor did not return image_sizes/pixel_values")
    image_sizes = inputs["image_sizes"]
    image_sizes = image_sizes.tolist() if hasattr(image_sizes, "tolist") else image_sizes
    prompt_tokens = int(inputs["input_ids"].shape[1])
    processed = time.perf_counter()
    progress.update("transfer to GPU")
    inputs = inputs.to("cuda:0")
    # Match the model's non-quantized vision compute dtype on T4.
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype=torch.float16)
    torch.cuda.synchronize()
    transferred = time.perf_counter()
    progress.update("generation / first-token prefill")
    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
            use_cache=True, return_dict_in_generate=False,
            stopping_criteria=[progress.token_observer(prompt_tokens)])
    torch.cuda.synchronize()
    generated_at = time.perf_counter()
    progress.update("decode output")
    token_ids = generated[0, prompt_tokens:].detach().cpu().tolist()
    text = processor.tokenizer.decode(token_ids, skip_special_tokens=True,
                                      clean_up_tokenization_spaces=False)
    eos = model.generation_config.eos_token_id
    eos = {eos} if isinstance(eos, int) else set(eos or [])
    truncated = len(token_ids) >= max_new_tokens and (not token_ids or token_ids[-1] not in eos)
    finished = time.perf_counter()
    memory = {"peak_allocated_bytes": int(torch.cuda.max_memory_allocated()),
              "peak_reserved_bytes": int(torch.cuda.max_memory_reserved())}
    result = {
        "text": text, "raw_response": text, "lines": [],
        "layout": {"method": "vlm_prompt", "tables": [], "has_region_annotations": False},
        "image_size": list(image.size), "processed_image_sizes": image_sizes,
        "input_tokens": prompt_tokens, "generated_tokens": len(token_ids),
        "truncated": truncated, "finish_reason": "length" if truncated else "eos",
        "gpu_memory": memory,
        "timing_seconds": {"decode": decoded - started, "preprocess": processed - decoded,
                           "transfer": transferred - processed,
                           "generation": generated_at - transferred,
                           "decode_output": finished - generated_at, "total": finished - started},
    }
    del inputs, generated
    return result


## 4. Warm-up và nhận dạng

Warm-up dùng ảnh đầu tiên và giới hạn 64 token mỗi lần sinh; kết quả warm-up không được chấm.
Mỗi ảnh có log `START`, giai đoạn và số token mỗi 30 giây. Số token tăng cho biết vòng sinh đang tiến triển.
Giữ nguyên cấu hình khi tiếp tục: chạy lại ô benchmark sẽ bỏ qua các ID đã có checkpoint.
Lỗi đã lưu được giữ lại để báo cáo; dùng tên lượt mới nếu cần chạy lại toàn bộ.

`TOKEN LIMIT` nghĩa là văn bản có thể chưa đủ; bản chép dở vẫn tham gia CER/WER.
Khi đổi prompt, checkpoint, giới hạn ảnh hoặc token, đổi `RUN_NAME` để không trộn cấu hình.
Nếu cần dừng, bấm Stop và chạy ô tải ZIP. Không Disconnect/Delete runtime trước khi lưu kết quả.


In [ ]:
print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("Output:", OUTPUT_DIR, flush=True)
print("max_new_tokens:", MAX_NEW_TOKENS, "| max_image_side:", MAX_IMAGE_SIDE, flush=True)
print("Log every 30 seconds during each image; completed images are resumed from checkpoints.",
      flush=True)
torch.cuda.synchronize()
warmup_started = time.perf_counter()
warmup_preview = predict_image(samples[0], max_new_tokens=WARMUP_NEW_TOKENS)
torch.cuda.synchronize()
warmup_seconds = time.perf_counter() - warmup_started
print("Warm-up:", round(warmup_seconds, 2), "seconds")
print("Preview (not scored):", warmup_preview["text"][:250])
del warmup_preview
gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
atomic_json(OUTPUT_DIR / "manifest.json", manifest)
metadata = run_benchmark(
    samples, OUTPUT_DIR, identity,
    {"initialization_seconds": initialization_seconds, "warmup_seconds": warmup_seconds,
     "progress_monitor": "30s heartbeat + token observer; does not change generation settings"},
    predict_image)
print(json.dumps({k: metadata.get(k) for k in (
    "status", "completed_images", "failed_images", "pending_images", "truncated_images",
    "wall_seconds", "latency_p50_seconds", "latency_p95_seconds",
    "peak_gpu_allocated_bytes")}, indent=2))


### Kiểm tra khi nghi bị dừng

Chỉ chạy ô dưới sau khi đã dừng ô benchmark; một ô Python mới thường phải chờ ô đang chạy.
Đây là kiểm tra trạng thái GPU và checkpoint, không tiếp tục inference.
Colab miễn phí có hạn mức GPU thay đổi; notebook không gọi API tính phí theo token.
Chỉ xem xét hạn mức khi Colab hiển thị thông báo tương ứng.
[FAQ Colab](https://research.google.com/colaboratory/faq.html#resource-limits).


In [ ]:
try:
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total",
         "--format=csv"], check=False, timeout=15)
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print("Cannot query GPU:", exc)
run_file = OUTPUT_DIR / "run.json"
if run_file.exists():
    snapshot = json.loads(run_file.read_text(encoding="utf-8"))
    print("Saved status:", {key: snapshot.get(key) for key in
          ("status", "completed_images", "failed_images", "pending_images", "truncated_images")})
    records = load_records(OUTPUT_DIR, samples, snapshot["run_fingerprint"])
    for sample in samples:
        row = records.get(sample["sample_id"])
        if row is not None:
            print(sample["sample_id"], row["status"],
                  "seconds=", round(row["timing_seconds"]["total"], 1),
                  "tokens=", row.get("generated_tokens"),
                  "truncated=", row.get("truncated"))
        else:
            print("First pending image:", sample["sample_id"])
            break
else:
    print("No saved run.json at", OUTPUT_DIR)


## 5. Tải kết quả về máy

ZIP chứa `outputs/<tên lượt chạy>/`, không chứa ảnh hay trọng số. Thời gian tải trọng số/ảnh và warm-up
không cộng vào latency. `run.json` ghi riêng khởi tạo/warm-up theo session; `wall_seconds` cộng thời gian
vòng xử lý các session, gồm ghi checkpoint. Bộ nhớ GPU là đỉnh PyTorch allocated/reserved, không phải
toàn bộ VRAM theo nvidia-smi; chưa đo đỉnh RAM CPU. CPU và GPU cần báo cáo riêng khi so sánh tốc độ.


In [ ]:
import shutil
from google.colab import files

if not (OUTPUT_DIR / "run.json").exists():
    raise RuntimeError("Chưa có lượt chạy để tải")
saved_run = json.loads((OUTPUT_DIR / "run.json").read_text(encoding="utf-8"))
saved_records = load_records(OUTPUT_DIR, samples, saved_run["run_fingerprint"])
write_artifacts(OUTPUT_DIR, samples, saved_records, saved_run)
if saved_run["pending_images"] or saved_run["failed_images"]:
    print("Lượt chạy còn thiếu/lỗi:", saved_run["pending_images"], saved_run["failed_images"])
if saved_run["truncated_images"]:
    print("Ảnh chạm giới hạn token:", saved_run["truncated_images"])
archive = shutil.make_archive(
    str(Path("/content") / RUN_NAME), "zip", root_dir=WORK_ROOT,
    base_dir="outputs/" + RUN_NAME)
print("ZIP:", archive)
files.download(archive)


## 6. Chấm trên máy local

Tại `D:\2tuan`, kích hoạt `.venv`, giải nén ZIP vào repo để có thư mục
`outputs/ministral3_3b_colab_t4_nf4/`, rồi chạy:

```powershell
. .\.venv\Scripts\Activate.ps1
Expand-Archive -LiteralPath "$HOME\Downloads\ministral3_3b_colab_t4_nf4.zip" -DestinationPath .
python scripts/ocr_baseline.py score --run outputs/ministral3_3b_colab_t4_nf4 --output outputs/ministral3_3b_colab_t4_nf4/scores_py312.json
```

Sửa đường dẫn ZIP theo vị trí tải thực tế. Nếu đã có thư mục kết quả hoặc file điểm cùng tên,
dùng tên mới để giữ tách biệt các lượt chạy. Lệnh chấm dùng **ground truth local đã duyệt**,
không dùng nhãn cũ trên GitHub. Không chạy `draft` để thay ground truth.

Notebook đã được kiểm tra cú pháp và định dạng xuất/chấm bằng kiểm thử local; chưa thực thi
inference trên T4 trong môi trường phát triển. Sau lượt đầu, kiểm tra GPU, lỗi và số ảnh bị cắt
trong `run.json` trước khi đưa số liệu vào báo cáo.
